# Engineering Department Report Generator

In [ ]:
# S. Shirk
# V. Scalfani
# K. Walker
# G. Mansfield

**Outline:**  
- Imports/General Setup
- Main Data Collection
- Where Faculty are Publishing
- Open Access Data
- Referenced Works
- Funders
- Author Collaboration
- Analysis of Author Keywords
- Create the Automated Report

## What is the data in the report?

- Known Publication Outlets: These are the journals, conferences, or other sources where OpenAlex identifies that authors are publishing their work.

- Most Cited References: These are the sources of works that authors have cited in their publications.

  - To explain further: When an author cites a work, OpenAlex records a link to that cited work in a reference field. By tracing these links, we determine the original source of each reference—whether it’s a journal, article, or other publication. We aggregate all these sources to compile the list of Most Cited Journals.

- Sponsored Research: Who is funding publications on a per article basis.

- Open Access: What levels of access are publications afforded.

- Author Supplied Keywords: Keywords from publications to show current research trends.

- Author Collaborations: who faculty collaborate with on publications within and outside the department.

**Fill in the blanks**  

* There are a couple of variables that need to be filled for the report to give you want you want.  

* I have provided the data collection variables in the `General Setup` section and the report variables in the `Create the Automated Report` section. Change the variable values as need be. 

**Python Libraries**

Below are the licenses for the respective libraries used

- [Pandas (2.2.3)](github.com/pandas-dev/pandas/blob/main/LICENSE)
- [Matplotlib (3.10.1)](https://matplotlib.org/stable/project/license.html)
- [networkx (3.3)](https://github.com/networkx/nx-guides/blob/main/LICENSE)
- [wordcloud (1.9.3)](https://github.com/timdream/wordcloud/blob/main/MIT-LICENSE.txt)
- [jinja2 (3.1.5)](https://github.com/pallets/jinja/blob/main/LICENSE.txt)
- [weasyprint (62.3)](https://github.com/Kozea/WeasyPrint/blob/main/LICENSE)

## Imports


In [ ]:
import pandas as pd
import time
import requests
import matplotlib.pyplot as plt
import weasyprint
import os
import networkx as nx
import asyncio
import numpy as np
import re
from datetime import datetime, timedelta
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.transforms import Bbox
from collections import Counter
from wordcloud import WordCloud
from jinja2 import Environment, FileSystemLoader

## General Setup
The code is set up to use a csv called `author_ids.csv` in the `data` folder. Run the code block below to make all the folders you'll need.  

After making the folders, add your `authors_ids.csv` to the `data` folder. The structure of `author_ids.csv` should be as follows: 
 
author_ids,last_name,first_name,department,add_department

In [ ]:
if not os.path.exists('data'):
    os.makedirs('data')
if not os.path.exists('imgs'):
    os.makedirs('imgs')
if not os.path.exists('html'):
    os.makedirs('html')

In [ ]:
author_ids_csv = pd.read_csv('data/author_ids.csv')
departments = author_ids_csv['department'].tolist()
departments = set(departments)

# List of departments to analyze
departments

### Data Collection Variables

In [ ]:
# Update to focus on desired institution here
place = 'The University of Alabama'

# Update to focus on desired academic department here
department = ''

# Set the lower bound year (inclusive) for publication filtering
# Only works with publication_year >= this value will be included in the analysis
year = 2013

# API Authentication
OpenAlex now requires an API Key: https://developers.openalex.org/api-reference/introduction


In [ ]:
from dotenv import load_dotenv
load_dotenv()

OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY")
if not OPENALEX_API_KEY:
    raise ValueError("OPENALEX_API_KEY not found in .env file")

API_PARAMS = {
    "api_key": OPENALEX_API_KEY
}

## Main Data Collection
The code is set up to use a csv called `author_ids.csv` in the `data` folder.

In [ ]:
author_ids_csv = pd.read_csv('data/author_ids.csv')
author_ids = author_ids_csv[author_ids_csv['department'] == department]['author_ids'].tolist()

author_ids

# Count the number of author_ids in the specified department
author_ids_count = len(author_ids)
print(f"Number of author_ids in {department}: {author_ids_count}")

In [ ]:
def get_author_data(author_ids):
    authors_list = []
    for author_id in author_ids:
        author_id_clean = author_id.split("/")[-1]
        base_url = f"https://api.openalex.org/authors/{author_id_clean}"
        try:
            response = requests.get(base_url)
            response.raise_for_status()

            author_data = response.json()

            author_info = {
                "name": author_data.get('display_name'),
                "id": author_data.get('id'),
                "orcid": author_data.get('orcid'),
                "works_count": author_data.get('works_count'),
                "works_api_url": author_data.get('works_api_url'),
            }

            authors_list.append(author_info)

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred for {author_id}: {http_err}")
        except Exception as err:
            print(f"An error occurred for {author_id}: {err}")
        
    return authors_list

no_source = []

def get_works_data(author_ids):
    works_data_list = []

    for author_id in author_ids:
        base_url = (
            "https://api.openalex.org/works"
            f"?filter=authorships.author.id:{author_id}"
            "&select=id,doi,publication_year,primary_location,type,open_access,"
            "authorships,keywords,topics,awards,referenced_works"
            "&per_page=100"
        )

        cursor = "*"

        try:
            while cursor:
                response = requests.get(f"{base_url}&cursor={cursor}")
                response.raise_for_status()

                works_data = response.json()
                works_info = works_data.get("results", [])

                for work in works_info:
                    try:
                        if (
                            work["publication_year"] > year
                            and work["type"] == "article"
                            and work["primary_location"]["source"] is not None
                        ):
                            for author in work["authorships"]:
                                if (
                                    author.get("institutions")
                                    and author["institutions"][0]["display_name"] == "University of Alabama"
                                ):
                                    works_data_list.append(work)
                                    break

                        elif (
                            work["publication_year"] > year
                            and work["type"] == "article"
                            and work["primary_location"]["source"] is None
                        ):
                            authors = []
                            for author in work["authorships"]:
                                if author["author"]["display_name"] in authors_list_df["name"].tolist():
                                    authors.append(author["author"]["display_name"])
                            no_source.append([work["doi"], authors])

                    except Exception:
                        continue

                cursor = works_data.get("meta", {}).get("next_cursor")

                time.sleep(0.2)

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred for {author_id}: {http_err}")
        except Exception as err:
            print(f"An error occurred for {author_id}: {err}")

    return works_data_list

def get_oa_data(works_data):
    oa_data = []
    for work in works_data:
        if work.get('open_access'):
            oa_info = {
                "is_oa": work['open_access'].get('is_oa'),
                "oa_status": work['open_access'].get('oa_status'),
                "oa_url": work['open_access'].get('oa_url'),
            }
            oa_data.append(oa_info)
    return oa_data

def get_keywords_and_sources(works_data):
    concept_keywords = []
    sources_keywords = []

    for work in works_data:
        # Use topics instead of deprecated concepts
        if work.get('topics'):
            for topic in work['topics']:
                name = topic.get('display_name')
                if name:
                    concept_keywords.append(name)

        # Venue/source from primary_location.source
        if work.get('primary_location'):
            try:
                source_name = work['primary_location']['source']['display_name']
                if source_name:
                    sources_keywords.append(source_name)
                else:
                    sources_keywords.append('No source')
            except:
                sources_keywords.append('No source')
        else:
            sources_keywords.append('No source')

    return concept_keywords, sources_keywords

# Author Data
authors_list = get_author_data(author_ids)
authors_list_df = pd.DataFrame(authors_list)

# Work Data
author_ids = [author_id.split("/")[-1] for author_id in authors_list_df["id"].tolist()]
works_data_list = get_works_data(author_ids)
works_data_df = pd.json_normalize(works_data_list)

# De-duplicate works so the same publication is only counted once
# when multiple department authors are attached to the same work.
works_data_df = works_data_df.drop_duplicates(subset=["id"]).reset_index(drop=True)

seen_ids = set()
works_data_deduped = []

for work in works_data_list:
    work_id = work.get("id")
    if work_id not in seen_ids:
        works_data_deduped.append(work)
        seen_ids.add(work_id)

# Total number of documents
total_docs = works_data_df.shape[0]
print(f"Total number of documents after filters: {total_docs}")

# Open Access data
oa_data_list = get_oa_data(works_data_deduped)
oa_data_df = pd.DataFrame(oa_data_list)

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 50)

# Concept keywords and sources
concept_keywords, sources_keywords = get_keywords_and_sources(works_data_deduped)

authors_list_df

In [ ]:
# oa_data_df
# sources_keywords
works_data_df
works_data_df.to_csv(f'data/{department}_works_data.csv', index=False)


In [ ]:
no_source_df = pd.DataFrame(no_source, columns=['doi', 'authorships'])
no_source_df

## Where Faculty are Publishing
This is the Known Publication Outlets  

In [ ]:
# Initialize Counter for frequency counting
source_freq = Counter()

# Count undefined sources for note
no_source_publication_count = 0

# Iterate over each source in sources_keywords
for source in sources_keywords:
    if source is not None and isinstance(source, str):
        journal = source.strip()

        if journal == 'No source':
            no_source_publication_count += 1
        elif journal:
            source_freq.update([journal])

# Convert Counter to a sorted list of (keyword, frequency) tuples in descending order of frequency
source_freq_list = sorted(source_freq.items(), key=lambda item: item[1], reverse=True)[:10]

# Unzip sorted list into separate sources and frequencies for DataFrame creation
sources, frequencies = zip(*source_freq_list)

# Create DataFrame from the sorted lists
publishing_df = pd.DataFrame({
    'Sources': sources,
    'Frequency': frequencies
})
publishing_df['Frequency'] = publishing_df['Frequency'].apply(lambda x: f"{x:,}")

publishing_df

## Open Access Data

In [ ]:
oa_status_count = {
        "closed": 0,
        "diamond": 0,
        "gold": 0,
        "green": 0,
        "hybrid": 0,
        "bronze": 0,
        "unknown": 0,
}

for oa_status in oa_data_df['oa_status']:
    oa_status_count[oa_status] += 1

oa_status_count = {key: value for key, value in oa_status_count.items() if value != 0}
# oa_status_count = {key: value for key, value in oa_status_count.items()}

# Calculate total and closed access percentage
total_articles = sum(oa_status_count.values())
closed_articles = oa_status_count.get("closed", 0)

percent_closed = round((closed_articles / total_articles) * 100) if total_articles > 0 else 0

# Make a DataFrame from the dictionary with columns "status" and "count"
open_access_df = pd.DataFrame(oa_status_count.items(), columns=["Status", "Count"])
open_access_df['Count'] = open_access_df['Count'].apply(lambda x: f"{x:,}")

open_access_df

## Referenced Works
This is what published articles are citing within themselves. 
* The first code block will call the API A LOT. This will take a long time to run. Took me about 2 hours and 15 minutes to run this script for about 17,000 refererences. <br><br>
* The second code block will write a csv with a list of all the sources gathered by the API for all of our references. <br><br>
* You should only have to run those code blocks once for each department.

In [ ]:
# Get number of references found in the works data
referenced_works = []

for work in works_data_deduped:
    referenced_works.append(work['referenced_works'])

referenced_works_ids = []
for ref_list in referenced_works:
    for ref in ref_list:
        referenced_works_ids.append(ref.split('/')[3])

len(referenced_works_ids)

In [ ]:
# # Comment/uncomment section to run/not run

# # Batch retrieval of referenced works metadata.
count = 0

async def main():
    target = datetime.now().replace(hour=0, minute=1, second=0, microsecond=0)

    if target < datetime.now():
        target += timedelta(days=1)

    now = datetime.now()
    delay = (target - now).total_seconds()
    if delay > 0:
        print(f"Waiting {delay / 3600:.2f} hours until {target.strftime('%Y-%m-%d %H:%M:%S')}...")
        await asyncio.sleep(delay)

    print("Time reached! Starting API calls...")

    count = 75000

    for ref in referenced_works_ids[75000:]:
        base_url = f"https://api.openalex.org/works/{ref}"
        print(count)
        count += 1

        try:
            response = requests.get(base_url)
            response.raise_for_status()

            work_data = response.json()
            if work_data.get('primary_location'):
                try:
                    if work_data['primary_location']['source']['type'] == 'journal':
                        source_names.append(work_data['primary_location']['source']['display_name'])
                except:
                    source_names.append('No source')
            else:
                source_names.append('No source')
            time.sleep(0.2)

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred for {ref}: {http_err}")
        except Exception as err:
            print(f"An error occurred for {ref}: {err}")

# Batch retrieval of referenced works metadata (100 IDs per request)
# Uses OpenAlex list+filter endpoint with OR ("|") syntax to query multiple work IDs simultaneously.
# This approach reduces API call volume by ~100x compared to sequential single-ID requests,
# improving performance and scalability for large reference lists.
# Returned records are used to extract source (journal) information for downstream analysis.

from collections import Counter
import requests

def chunk_list(seq, size=100):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

def get_source_names_for_references(referenced_works_ids, api_key=None):
    source_names = []
    session = requests.Session()

    for chunk in chunk_list(referenced_works_ids, 100):
        ids_str = "|".join(chunk)
        params = {
            "filter": f"openalex:{ids_str}",
            "select": "id,primary_location",
            "per_page": 100
        }
        if api_key:
            params["api_key"] = api_key

        response = session.get("https://api.openalex.org/works", params=params)
        response.raise_for_status()
        results = response.json().get("results", [])

        for work_data in results:
            primary_location = work_data.get("primary_location")
            if primary_location and primary_location.get("source"):
                source = primary_location["source"]
                if source.get("type") == "journal":
                    source_names.append(source.get("display_name", "No source"))
                else:
                    source_names.append("No source")
            else:
                source_names.append("No source")

    return source_names

source_names = get_source_names_for_references(referenced_works_ids, api_key=OPENALEX_API_KEY)

In [ ]:
# # # Comment/uncomment section to run/not run

reference_data = pd.DataFrame({
    'References': source_names,
})

reference_data.to_csv(f'data/{department}_references.csv', index=False)

In [ ]:
reference_freq = Counter()

reference_data = pd.read_csv(f'data/{department}_references.csv')

# Iterate over each reference in work_data_df
for reference in reference_data['References']:
    if reference is not None and isinstance(reference, str):
        # Split the string by ',' and clean up the words, with a different variable name in list comprehension
        keywords = [kw.strip() for kw in reference.split(',') if kw.strip() and kw.strip() != 'No source']
        
        # Update reference frequency counter with clean keywords
        reference_freq.update(keywords)

# Convert Counter to a sorted list of (keyword, frequency) tuples in descending order of frequency
reference_freq_list = sorted(reference_freq.items(), key=lambda item: item[1], reverse=True)[:10]

# Unzip sorted list into separate references and frequencies for DataFrame creation
references, frequencies = zip(*reference_freq_list)

# Create DataFrame from the sorted lists
references_df = pd.DataFrame({
    'References': references,
    'Frequency': frequencies
})
references_df['Frequency'] = references_df['Frequency'].apply(lambda x: f"{x:,}")

references_df

## Funders

In [ ]:
# Collect all funder_display_name values
funder_names = []

for work in works_data_deduped:
    if 'awards' in work and isinstance(work['awards'], list):
        for grant in work['awards']:
            name = grant.get('funder_display_name')
            if name:
                funder_names.append(name)

# Count the occurrences of each full display name
funder_counts = Counter(funder_names)

# Get top 7 funders
top_n = 7
sorted_funders = funder_counts.most_common()
top_funders = sorted_funders[:top_n]
other_funders = sorted_funders[top_n:]

# Calculate frequency for 'Other'
other_freq = sum(freq for _, freq in other_funders)
# original: top_funders.append(('Other', other_freq))
top_funders.append((f"Other", other_freq))

# Prepare DataFrame
funders, frequencies = zip(*top_funders)
colors = ['#9E1B32', '#772432', '#ffdc81', '#c21e3f', '#DAD7CB', '#ab8f54', "#8F8F91", "#7e6870"]
empty = [' '] * len(funders)

funders_df = pd.DataFrame({
    'Color': empty,
    'Funders': funders,
    'Frequency': frequencies,
})
funders_df['Frequency'] = funders_df['Frequency'].apply(lambda x: f"{x:,}")

def highlight_color(row):
    color = colors[(row.name) % len(colors)]
    return [f'background-color: {color};' if col == 'Color' else '' for col in row.index]

# Print how many funder names were grouped into "Other"
print(f"{len(other_funders)} funders grouped into 'Other'")

# Display
# funders_df.style.apply(highlight_color, axis=1)
funders_df.style


In [ ]:
# Create a pie chart with the funders data

fig, ax = plt.subplots(figsize=(12, 6))
# ax.pie(funders_df['Frequency'], autopct='%1.1f%%', startangle=90, colors=colors)
ax.pie(funders_df['Frequency'].apply(lambda x: int(str(x).replace(',', ''))), autopct='%1.1f%%', startangle=90, colors=colors)
ax.axis('equal')
plt.tight_layout(rect=[0, 0, 0.5, 1])
plt.show()

fig_width, fig_height = fig.get_size_inches()
bbox = Bbox.from_bounds(0, 0, 0.5 * fig_width, fig_height)

fig.savefig('imgs/funders_pie_chart.png', bbox_inches=bbox)

## Author Collaboration

In [ ]:
connections = []
for work in works_data_deduped:
    work_id = work['id']
    author_id = []
    name = []
    if work.get('authorships'):
        for authorship in work['authorships']:
            author_id.append(authorship['author']['id'])
            name.append(authorship['author']['display_name'])
            institutions = []
            if authorship.get('institutions'):
                for institution in authorship['institutions']:
                    institutions.append(institution['display_name'])
    connections_data = (work_id, author_id, name, institutions)        
    connections.append(connections_data)

connections_df = pd.DataFrame(connections, columns=['Work_ID', 'Author IDs', 'Author Names', 'Institutions'])
connections_df

# Create a graph
G = nx.Graph()

# Add edges to the graph
for _, row in connections_df.iterrows():
    authors = row['Author Names']
    for i in range(len(authors)):
        for j in range(i + 1, len(authors)):
            if ((authors[i] in authors_list_df['name'].values or authors[j] in authors_list_df['name'].values)):
                if "University of Alabama" in row['Institutions']: # put institution after if and before in
                    G.add_edge(authors[i], authors[j])

# Highlight nodes that are in authors_list_df['name']
highlighted_authors = set(authors_list_df['name'].values)

# Assign colors to nodes
node_colors = [
    '#9E1B32' if node in highlighted_authors else '#828A8F'
    for node in G.nodes()
]


# Draw the graph
plt.figure(figsize=(15, 8))
pos = nx.spring_layout(G, k=0.2)

# Adjust label positions to be slightly above the nodes
label_pos = {node: (coords[0], coords[1] + 0.03) for node, coords in pos.items()}

nx.draw(
    G, pos, with_labels=False,
    node_size=100, font_size=15,
    node_color=node_colors, edge_color='gray'
)

# Draw labels separately
nx.draw_networkx_labels(
    G, label_pos,
    labels={node: node if node in highlighted_authors else '' for node in G.nodes()},
    font_size=15
)

plt.gca().margins(0, 0)
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

plt.savefig('imgs/author_graph.png', bbox_inches='tight')
plt.show()

# Get all faculty who actually appear in the graph (i.e., have at least one collaboration)
faculty_names_in_graph = set(node for node in G.nodes if node in authors_list_df['name'].values)

# Get degree (number of unique collaborators) for each faculty in the graph
degrees = [(node, deg) for node, deg in G.degree() if node in faculty_names_in_graph]

# Only calculate median if there are any faculty nodes
if degrees:
    median_collaborators = int(np.median([deg for _, deg in degrees]))
    num_faculty = len(degrees)
else:
    median_collaborators = 0
    num_faculty = 0

# Sort by number of collaborators, descending, and take top 5
top5_collaborators = sorted(degrees, key=lambda x: x[1], reverse=True)[:5]

# Prepare an HTML string for the report
top5_collab_str = (
    f"<p>Median number of collaborators: {median_collaborators}<br>"
    f"Number of faculty counted: {num_faculty}</p>"
)

name, degree = zip(*top5_collaborators) if top5_collaborators else ([], [])

# Create a DataFrame using these sorted lists
data = {
    'Name': name,
    'Degree': degree
}
collab_df = pd.DataFrame(data)

print(collab_df)
print("Median number of collaborators (faculty only):", median_collaborators)
print(f"Number of faculty counted: {num_faculty}")


## Analysis of Author Keywords

In [ ]:
# Create a wordcloud and table of most frequent author keywords
keyword_counter = Counter()

in_colormap = LinearSegmentedColormap.from_list("colors", ["#9e1b32", "#9e1b32", "#828a8f", "#772432", "#f2d653"])

for keyword_string in concept_keywords:

    if keyword_string is not None and type(keyword_string) == str:
        # Split the string by ',' and clean up the words
        keywords = [keyword.strip().lower() for keyword in keyword_string.split(',')]

        # add keywords
        keyword_counter.update(keywords)

        # remove empty strings
        if '' in keyword_counter:
           del keyword_counter['']

# Convert the Counter to a list of (keyword, frequency) tuples and sort it by frequency in descending order
sorted_keyword_data = sorted(keyword_counter.items(), key=lambda item: item[1], reverse=True)[:5]

# Create a set of the top 5 keywords
top5_keywords = {keyword for keyword, freq in sorted_keyword_data}

# Create a filtered dictionary excluding the top 5 keywords for the word cloud
filtered_keyword_counter = {keyword: freq for keyword, freq in keyword_counter.items() if keyword not in top5_keywords}

# Generate the word cloud using the filtered frequencies
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap=in_colormap).generate_from_frequencies(filtered_keyword_counter)

# Split the sorted tuple list into two lists: one for keywords and one for their frequencies
keywords, frequencies = zip(*sorted_keyword_data)

# Create a DataFrame using these sorted lists
data = {
    'Keyword': keywords,
    'Frequency': frequencies
}
keywords_df = pd.DataFrame(data)
keywords_df['Frequency'] = keywords_df['Frequency'].apply(lambda x: f"{x:,}")
# Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.tight_layout()
plt.show()
print(keywords_df)

wordcloud.to_file('imgs/author_keywords_wordcloud.png')

## Create the Automated Report

### Report Variables

In [ ]:
# ------------------------------------------------------------
# Text 1. Fill in 1st paragraph text here. 
# ------------------------------------------------------------
text1 = f'''            
    This report provides an overview of research activity in the department of {department} at {place}, analyzing {total_docs:,} scholarly articles published since {year} from {author_ids_count} current faculty represented. 
    Report data were gathered via the OpenAlex API and analyzed using custom Python code to identify key trends in publications, funding sources, and research collaborations.
    It highlights top publication outlets, cited references, sponsored research, and open access trends. The report also includes a word cloud of common research themes and a network graph of co-authorships,
    revealing key areas of focus and collaboration.
'''
# ------------------------------------------------------------
# Text 2. Fill in 2nd paragraph text here. 
# ------------------------------------------------------------
text2 = f'''
    Around {percent_closed}% of the department’s research remains behind paywalls, limiting public access and reuse. While Gold and Hybrid open access options are limited, many authors can legally deposit versions of their work 
    in our Institutional Repository. This cost-free route improves visibility, meets funder requirements, and extends impact. The library offers guidance on self-archiving and publisher policies.
'''
# ------------------------------------------------------------
# Text 3. Fill in text for underneath the network here.
# ------------------------------------------------------------
network_text = f'''
    The word cloud shows common research themes based on author supplied keywords, while the network graph has each node as a unique author. The nodes that are labeled red are faculty within the department, while the grey nodes are co-authorships beyond the department. 
'''
# ------------------------------------------------------------
# You may have to shorten the text of your paragraphs 
# to maintain the formatting of the PDF.
# ------------------------------------------------------------

In [ ]:
import platform

if platform.system() != "Windows":
    print("Running on Linux or MacOS")
    font_size = 12
else:
    font_size = 14

def generate_html_report(template_path, context, output_html):
    env = Environment(loader=FileSystemLoader(os.path.dirname(template_path)))
    template = env.get_template(os.path.basename(template_path))
    html_content = template.render(context)

    with open(output_html, 'w', encoding='utf-8') as f:
        f.write(html_content)

def convert_html_to_pdf(input_html, output_pdf):
    weasyprint.HTML(input_html).write_pdf(output_pdf)

pie_table_html = (
    funders_df.reset_index(drop=True)
        .style.apply(highlight_color, axis=1)
        .hide(axis="index")
        .to_html()
)

# Replace the first table id with your own
pie_table_html = re.sub(r'<table id="[^"]+"', '<table id="funders-table"', pie_table_html, count=1)

def generate_report(filename):
    # Create context for the template
    context = {
        'title': f'Department of {department} Research Overview',
        'text': f'{text1}',
        'publishing_table': publishing_df.to_html(index=False),
        'references_table': references_df.to_html(index=False),
        'open_access_table': open_access_df.to_html(index=False),
        'keywords_wordcloud': keywords_df.to_html(index=False),
        'pie_table': pie_table_html,
        'image_path': '../imgs/author_keywords_wordcloud.png',
        'chart_image_path': '../imgs/funders_pie_chart.png',
        'top5_collab_str': top5_collab_str,
        'collab_df': collab_df.to_html(index=False),

        'author_network_path': '../imgs/author_graph.png',
        'text2': f'{text2}',
        'other_funders': f"'Other' category consist of {funders_df.iloc[-1]['Frequency']} awards from {len(other_funders)} different funders.",
        'network_text': f'{network_text}',
        'font_size': font_size,
        'year': year,
    }

    # Paths for the files
    template_path = f'html/{department}_report_template.html'
    output_html = f'html/{department}_report.html'
    output_pdf = filename

    # Generate HTML report
    generate_html_report(template_path, context, output_html)

    # Convert HTML to PDF
    convert_html_to_pdf(output_html, output_pdf)

# Create an HTML template
template_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{{ title }}</title>
    <style>
        @page {
            margin: 0.5in;
        }
        body {
            font-family: Arial, sans-serif;
            font-size: {{ font_size }}px;
            margin: 0;
            margin-left: auto;
            margin-right: auto;
            padding: 0;
        }
        p {
            text-indent: 0px;
            text-align: justify;
        }
        footer {
            position: fixed;
            bottom: 0;
            left: 0;
            right: 0;
            text-align: right;
            font-size: 10px;
            color: #999;
        }
        h1 {
            color: #333;
            width: 80%;
            text-align: left;
            margin-top: 0px;
            padding: 0px;
            margin-bottom: 0px;
        }
        h3 {
            width: 100%;
            margin-bottom: 5px;
            margin-top: 0px;
            margin-left: auto;
            margin-right: auto;
        }
        .container {
            display: flex;
            flex-wrap: nowrap;
            box-sizing: border-box;
            page-break-inside: avoid;
        }
        .column {
            flex: 1;
            width: 50%;
            padding: 15px;
            padding-top: 0px;
            box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
            background-color: #fff;
            border-radius: 5px;
            box-sizing: border-box;
            page-break-inside: avoid;
        }
        .pie .column {
            flex: 1;
            width: 100%;
            padding: 15px;
            padding-top: 0px;
            box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
            background-color: #fff;
            border-radius: 5px;
            box-sizing: border-box;
            page-break-inside: avoid;
        }
        .wide-column {
            flex-grow: 2;
            width: 50%;
            padding-left: 15px;
        }

        .narrow-column {
            flex-grow: 1;
            width: 50%;
            padding-right: 15px;
        }
        .column img {
            max-width: 100%;
            height: auto;
            display: block;
            margin: 0;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-left: auto;
            margin-right: auto;
            margin-bottom: 20px;
            page-break-inside: auto;
        }
        th, td {
            padding: 1px;
            text-align: left;
            page-break-inside: avoid;
            font-size: 11px;
            word-wrap: break-word;
            white-space: normal;
        }
        th {
            background-color: #f0f0f0;
            font-weight: bold;
        }
            table td:nth-child(2), table th:nth-child(2) {
        text-align: right;
        }
            table td:nth-child(3), table th:nth-child(3) {
        text-align: right;
        }
        #funders-table td:nth-child(2),
        #funders-table th:nth-child(2) {
        text-align: left !important;
        }
        .box_image {
            position: absolute;
            right: 10px;
            top: 0px;
            padding-right: 0px;
            padding-left: 0px;
            z-index: 1000;
        }
        .box{ 
          width: 100%; 
          height: 200px;
        }
        .clearfix::after {
            content: "";
            clear: both;
            display: table;
        }
        .normal_table {
            width: 50%;
            border-collapse: collapse;
            margin-left: auto;
            margin-right: auto;
            margin-bottom: 20px;
            page-break-inside: auto;
        }
    </style>
</head>
<body>
    <div style="position: relative; width: 100%;">
        <h1>{{ title }}</h1>
        <div class="box_image">
            <img src="../imgs/UALIB_logo.png" alt="Logo" style="max-height: 35px;">
        </div>
    </div>
    <p>{{ text }}</p>

    <div class="container clearfix">
        <div class="column">

            <h3>Top 10 - Known Publication Outlets</h3>
            {{ publishing_table | safe }}
            <p><em>{{ publishing_note }}</em></p>

        </div>
        <div class="column">

            <h3>Top 10 - Most Cited References</h3>
            {{ references_table | safe }}
            <p><em>{{ references_note }}</em></p>

        </div>
    </div>

<!-- This block uses flexbox to center the pie chart and legend vertically -->

    <div class="container clearfix" style="display: flex; align-items: center; justify-content: space-between;">

        <div class="column" style="flex: 1; text-align: center;">

            <h3>Sponsored Research</h3>

            <img src="{{ chart_image_path }}" alt="Pie Chart" style="max-height: 300px; width: auto;">

            <h6 style="margin-top: 10px;">Disclaimer: The funds data gathered is based on a per article basis and not a per author basis.</h6>
        </div>

    <div class="column" style="flex: 1; text-align: center;">

        <div>

            <div style="margin-bottom: 10px;">{{ pie_table | safe }}</div>

            <h6 style="margin-top: 0px;">{{ other_funders }}</h6>

        </div>

    </div>

<!-- This block uses flexbox to center the pie chart and legend vertically -->

</div>


    <div class="container clearfix">
        <div class="column">

            <h3>Open Access Summary</h3>
            {{ open_access_table | safe }}

        </div>
        <div class="column">

            <p>{{ text2 }}</p>

        </div>
    </div>

    <h3>Author Supplied Keywords</h3>
    <img src="{{ image_path }}" alt="Word Cloud" style="width: 100%;">

    <!-- div class="normal_table" -->
    <div class="container clearfix">
        <div class="column">

            <h3 style="margin-bottom: 2px; padding-bottom: 0px;">Top 5 Author Keywords</h3>
            <h6 style="margin-top: 0px;">Excluded from the word cloud</h6>
            {{ keywords_wordcloud | safe }}
        </div>

        <div class="column">
            <h3 style="margin-bottom: 2px; padding-bottom: 28px;">Top 5 Author Collaborators & Median</h3>

        {{ collab_df | safe }}
        {{ top5_collab_str | safe }}
        
        </div>
    </div>
    <h3>Network Graph of Author Collaborations</h3>
    <img src="{{ author_network_path }}" alt="Author Network" style="width: 100%;">

  {{ network_text | safe }}

    <footer>

        All data was gathered from OpenAlex API from {{ year }} to present.

    </footer>

</body>
</html>
"""

# Save the template content to a file
with open(f'html/{department}_report_template.html', 'w') as f:
    f.write(template_content)

# Generate the report
generate_report(f'{department}_report.pdf')